# **Flight Fare Prediction Using Machine Learning**
Airlines and travel platforms want to estimate ticket prices based on route, airline, and travel date to help with pricing strategy and dynamic recommendations.

## **Problem Definition & Data Understanding**

#### **Machine Learning Task**

This project is formulated as a *supervised regression* problem.

- **Target (y):** Total Fare (BDT)

- **Inputs (X):** route and trip characteristics such as airline, source, destination, departure date/time (engineered into month/day features), duration, stopovers, class, booking source, seasonality, and days before departure.

#### **Success Criteria (Evaluation Metrics)**

We will evaluate models using:

- **`R²`** (**Coefficient of Determination**): how much variance in fare is explained by the model.

- **Mean Absolute Error** (**`MAE`**): average absolute prediction error (in BDT).

- **Root Mean Squared Error** (**`RMSE`**): error magnitude that penalizes large mistakes more (in BDT).


This notebook documents the workflow and results.

In [14]:
from pathlib import Path
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

In [15]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# Data paths
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Flight_Price_Dataset_of_Bangladesh.csv"
SILVER_PATH = PROJECT_ROOT / "data" / "interim" / "flight_fares_silver.parquet"
GOLD_PATH = PROJECT_ROOT / "data" / "processed" / "flight_fares_gold.parquet"

# Report/model artifacts 
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

print("Project root:", PROJECT_ROOT)
for p in [RAW_PATH, SILVER_PATH, GOLD_PATH, REPORTS_DIR, MODELS_DIR]:
    print(f"{p}  ->  {'exists' if p.exists() else 'missing'}")

Project root: /mnt/c/Users/Amalitech/Downloads/Arlette/Amali/DEMO9
/mnt/c/Users/Amalitech/Downloads/Arlette/Amali/DEMO9/data/raw/Flight_Price_Dataset_of_Bangladesh.csv  ->  exists
/mnt/c/Users/Amalitech/Downloads/Arlette/Amali/DEMO9/data/interim/flight_fares_silver.parquet  ->  exists
/mnt/c/Users/Amalitech/Downloads/Arlette/Amali/DEMO9/data/processed/flight_fares_gold.parquet  ->  exists
/mnt/c/Users/Amalitech/Downloads/Arlette/Amali/DEMO9/reports  ->  exists
/mnt/c/Users/Amalitech/Downloads/Arlette/Amali/DEMO9/models  ->  exists


### **Dataset**
We are using the Bangladesh Price dataset from kaggle: [Flight Price Dataset of Bangladesh](https://www.kaggle.com/datasets/mahatiratusher/flight-price-dataset-of-bangladesh)

To keep the workflow reproducible and production-oriented, the pipeline is organized into layers:

- **Raw (Bronze)**: original CSV (unchanged)
 
- **Silver**: typed + standardized data with validation/audit columns

- **Gold**: modeling-ready dataset

In [16]:
df_raw = pd.read_csv(RAW_PATH)
df_raw.head(3)

,Airline,Source,Source Name,Destination,Destination Name,Departure Date & Time,Arrival Date & Time,Duration (hrs),Stopovers,Aircraft Type,Class,Booking Source,Base Fare (BDT),Tax & Surcharge (BDT),Total Fare (BDT),Seasonality,Days Before Departure
0,Malaysian Airlines,CXB,Cox's Bazar Airport,CCU,Netaji Subhas Chandra Bose International Airpo...,2025-11-17 06:25:00,2025-11-17 07:38:10,1.219526,Direct,Airbus A320,Economy,Online Website,21131.225021,5169.683753,26300.908775,Regular,10
1,Cathay Pacific,BZL,Barisal Airport,CGP,"Shah Amanat International Airport, Chittagong",2025-03-16 00:17:00,2025-03-16 00:53:31,0.608638,Direct,Airbus A320,First Class,Travel Agency,11605.395471,200.000000,11805.395471,Regular,14
2,British Airways,ZYL,"Osmani International Airport, Sylhet",KUL,Kuala Lumpur International Airport,2025-12-13 12:03:00,2025-12-13 14:44:22,2.689651,1 Stop,Boeing 787,Economy,Travel Agency,39882.499349,11982.374902,51864.874251,Winter Holidays,83


In [17]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57000 entries, 0 to 56999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Airline                57000 non-null  object 
 1   Source                 57000 non-null  object 
 2   Source Name            57000 non-null  object 
 3   Destination            57000 non-null  object 
 4   Destination Name       57000 non-null  object 
 5   Departure Date & Time  57000 non-null  object 
 6   Arrival Date & Time    57000 non-null  object 
 7   Duration (hrs)         57000 non-null  float64
 8   Stopovers              57000 non-null  object 
 9   Aircraft Type          57000 non-null  object 
 10  Class                  57000 non-null  object 
 11  Booking Source         57000 non-null  object 
 12  Base Fare (BDT)        57000 non-null  float64
 13  Tax & Surcharge (BDT)  57000 non-null  float64
 14  Total Fare (BDT)       57000 non-null  float64
 15  Se

In [18]:
df_raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Airline,57000,24,US-Bangla Airlines,4496,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Source,57000,8,CGP,7241,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Source Name,57000,8,"Shah Amanat International Airport, Chittagong",7241,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Destination,57000,20,JED,3071,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Destination Name,57000,20,"King Abdulaziz International Airport, Jeddah",3071,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Departure Date & Time,57000,54126,2025-04-14 20:58:00,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Arrival Date & Time,57000,56944,2025-08-19 21:09:23,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Duration (hrs),57000.0,NaN,NaN,NaN,3.994955,4.094043,0.5,1.003745,2.644656,5.490104,15.831719
Stopovers,57000,3,Direct,36642,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Aircraft Type,57000,5,Airbus A320,23970,NaN,NaN,NaN,NaN,NaN,NaN,NaN



- The dataset contains `57,000 rows` with `17 columns` (mix of categorical, datetime, and numeric fare fields).

- **Key categorical fields** include Airline, Source, Destination, Class, Stopovers, and Booking Source.

- **Key numeric fields** include Duration (hrs), Days Before Departure, and the fare components: Base Fare (BDT), Tax & Surcharge (BDT), Total Fare (BDT).

- The departure timestamp has high granularity (many unique values), suggesting fares depend on time and booking lead time.

Now we validate data quality (missingness, duplicates, invalid values) and create a typed “Silver” dataset for reliable analysis.

In [19]:
bronze_report_path = PROJECT_ROOT/ Path("reports/bronze_quality.json")

if bronze_report_path.exists():
    bronze_report = json.loads(bronze_report_path.read_text(encoding="utf-8"))
    bronze_report
else:
    print("reports/bronze_quality.json not found. Run: python -m scripts.00_profile_raw")


Before transforming the data, we validate “Bronze” raw inputs for schema stability and obvious quality issues (missing values, duplicates, negative fares). This step prevents silent errors later in modeling.

In [ ]:
silver_report_path = Path("reports/silver_checks.json")

if silver_report_path.exists():
    silver_checks = json.loads(silver_report_path.read_text(encoding="utf-8"))
    silver_checks
else:
    print("reports/silver_checks.json not found. Run: python -m scripts.01_build_silver")


In [ ]:

imp = pd.read_csv("reports/permutation_importance.csv")
imp.head(10)


In [ ]:
import matplotlib.pyplot as plt

top_n = 8
top = imp.sort_values("importance_mean", ascending=False).head(top_n).copy()
top = top.sort_values("importance_mean", ascending=True)  # for nicer barh order

plt.figure(figsize=(8, 4.8))
plt.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
plt.title("Permutation Importance (Top Features)")
plt.xlabel("Mean importance (Δ performance when permuted)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


The model relies most on Duration (hrs), Aircraft Type, Class, and Destination, suggesting fare is driven mainly by trip characteristics and route/product differences.

Days Before Departure and Seasonality have smaller but meaningful influence.

Features with near-zero/negative importance add little predictive value in this model and may be redundant with stronger predictors.